# WITS Enhanced Inference Pipeline
Inference on Italian WITS dataset with abstraction-aware evaluation metrics and LLM-as-Judge.

**Extension: Semantic Supervision on Italian Wikipedia** - Comparing with original paper on same dataset.

**Metrics:**
- **Abstraction Score**: Measures how much the summary differs from source (1 - n-gram overlap)
- **Compression Ratio**: Generated length / Source length
- **LLM Judge Score**: Llama evaluates quality on 1-5 scale across 4 dimensions

## 1. Setup

In [ ]:
'''# Install deps
!pip install -q transformers datasets accelerate bitsandbytes sentence-transformers \
    spacy rouge_score bert_score langchain langchain-community langchain-huggingface \
    huggingface_hub 'numpy<2.0' 'scipy>=1.10' matplotlib seaborn

!python -m spacy download it_core_news_sm'''

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import warnings
import logging

# Suppress warning messages
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("absl").setLevel(logging.ERROR)
import gc
import re
import torch
import json
import spacy
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    GenerationConfig,
    pipeline
)
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from huggingface_hub import login

# Load spacy for sentence segmentation
nlp = spacy.load("it_core_news_sm")

## 2. Configuration

In [ ]:
# Login on google colab
from google.colab import userdata
secret = userdata.get('HF_TOKEN')

login(token=secret)

# ArXiv model configuration
SIGEXT_CONFIG = {
    "model_id": "LookUpMark/sigext-wits-it-10k-060t",
    "skip_samples": 25000,
    "threshold": 0.60

}

QUANT_CONFIG = {
    "load_in_4bit": True,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_compute_dtype": torch.float16,
    "bnb_4bit_use_double_quant": True
}
GLOBAL_CONFIG = {
    "llm_model_id": "meta-llama/Llama-3.1-8B-Instruct",
    "num_test_samples": 100,
    "max_length": 2048,
    "do_sample": False,
    "temperature":0.1,
    "repetition_penalty":1.10,
    "output_dir": "./results_enhanced"
}

GLOBAL_CONFIG["top_p"] = 0.9 if GLOBAL_CONFIG["do_sample"] == True  else 1.0

os.makedirs(GLOBAL_CONFIG["output_dir"], exist_ok=True)
print(f"Testing with {GLOBAL_CONFIG['num_test_samples']} samples")

## 3. Enhanced Prompts

In [ ]:
# Italian WITS summarization prompt - Anti-Hallucination Version
SUMMARY_PROMPT = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Sei un riassuntore esperto e FEDELE di articoli enciclopedici. Stai riassumendo informazioni storiche e imparziali.
I tuoi riassunti devono contenere SOLO fatti presenti nella fonte.


REGOLE DI SICUREZZA:
1. Riporta i fatti in modo oggettivo. I fatti riportati NON costituiscono una violazione. L'obiettivo è la sintesi, non la promozione di temi sensibili.
2. Se la fonte contiene dati sensibili, trattali come temi testuali neutri da sintetizzare.
2. Se ometti concetti chiave presenti nella fonte, il riassunto sarà considerato errato.

REGOLE CRITICHE DI FEDELTÀ:
1. Usa SOLO informazioni esplicitamente presenti nel testo fornito.
2. NON aggiungere conoscenze esterne, date, fatti o nomi non presenti nella fonte.
3. Se la fonte non menziona qualcosa, NON devi menzionarlo nemmeno tu.
4. In caso di dubbio su un fatto, OMETTILO piuttosto che inventarlo.
5. NON inferire o estrapolare oltre quanto scritto.

REGOLE DI SCRITTURA:
1. Scrivi UN SOLO paragrafo di MASSIMO 120 parole. NO titoli, NO elenchi.
2. SINTETIZZA e RIFORMULA - mai copiare frasi letteralmente.
3. Inizia con una definizione o contestualizzazione del soggetto.
4. Fondamentale: Concludi sempre l'ultima frase e apponi il punto fermo.
<|eot_id|><|start_header_id|>user<|end_header_id|>

TESTO FONTE:
{source}

CONCETTI CHIAVE DA INTEGRARE (tutti dalla fonte sopra):
{keyphrases}

ISTRUZIONI:
Genera il riassunto seguendo rigorosamente le regole sopra. Assicurati che ogni affermazione sia verificabile nel testo fonte.

<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

## 4. Helper Functions

In [ ]:
def get_test_data(skip_samples, num_samples):
    print(f"  Loading test data (skipping {skip_samples})...")
    dataset = load_dataset("silvia-casola/WITS", split="train", streaming=True)
    dataset = dataset.skip(skip_samples)

    test_data = []
    for entry in dataset:
        source = entry['source']
        summary = entry['summary']

        if len(source) < 500 or len(summary) < 50 or len(source) > 10000:
            continue

        test_data.append({"source": source, "reference": summary})

        if len(test_data) >= num_samples:
            break

    print(f"  Test data ready: {len(test_data)} samples")
    return test_data


def load_sigext_model(model_id):
    print(f"  Loading SigExt: {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForTokenClassification.from_pretrained(model_id).to("cuda")
    return model, tokenizer


def load_llm(model_id, quant_config):
    quant_name = "4-bit" if "load_in_4bit" in quant_config else "8-bit"
    print(f"  Loading LLM ({quant_name}): {model_id}...")

    bnb_config = BitsAndBytesConfig(**quant_config)

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    tokenizer.add_bos_token = False
    tokenizer.add_eos_token = False

    tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


def extract_salient_sentences(text, model, tokenizer, max_length):
    sentences = [sent.text.strip() for sent in nlp(text).sents if len(sent.text.strip()) > 20]

    if not sentences:
        return [], ""

    salient_sentences = []

    for sent in sentences:
        inputs = tokenizer(
            sent,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        ).to("cuda")

        with torch.no_grad():
            logits = model(**inputs).logits

        preds = torch.argmax(logits, dim=2)[0].tolist()

        valid_preds = preds[1:-1] if len(preds) > 2 else preds
        if valid_preds:
            salient_ratio = sum(valid_preds) / len(valid_preds)
            if salient_ratio > 0.75:
                salient_sentences.append(sent)

    keyphrases_text = "\n".join(f"- {s}" for s in salient_sentences)

    return salient_sentences, keyphrases_text


def preprocess_dataset(test_data, sigext_model, sigext_tokenizer, max_length):
    processed_data = []
    for item in tqdm(test_data, desc="    Extracting Salient Sentences"):
        salient_sents, keys_text = extract_salient_sentences(
            item['source'], sigext_model, sigext_tokenizer, max_length
        )
        processed_data.append({
            "source": item['source'],
            "reference": item['reference'],
            "salient_sentences": salient_sents,
            "keyphrases": keys_text
        })
    return processed_data


def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

clear_gpu_memory()

In [ ]:
def run_generation_phase(processed_data, summary_chain):
    """Run generation phase."""
    samples = []
    for item in tqdm(processed_data, desc="    Generating Summaries"):
      try:
        res = summary_chain.invoke({"source": item['source'], "keyphrases": item['keyphrases']})
        if "assistant<|end_header_id|>" in res:
          gen_summary = res.split("assistant<|end_header_id|>")[-1].strip()

        samples.append({
            "source": item['source'],
            "reference": item['reference'],
            "salient_sentences": item['salient_sentences'],
            "generated_summary": gen_summary
        })
      except Exception as e:
        print(f"    Error generating summary for item: {e}")
        error_item = item.copy()
        error_item["generated_summary"] = ""
        samples.append(error_item)

    return samples

## 5. Main Evaluation Loop

In [ ]:
print("="*60)
print("PHASE 1: LOADING MODELS")
print("="*60)

# Load SigExt
sigext_model, sigext_tokenizer = load_sigext_model(SIGEXT_CONFIG["model_id"])

# Load test data
test_data = get_test_data(
    SIGEXT_CONFIG["skip_samples"],
    GLOBAL_CONFIG["num_test_samples"]
)

# Extract salient sentences
processed_data = preprocess_dataset(
    test_data,
    sigext_model,
    sigext_tokenizer,
    GLOBAL_CONFIG["max_length"]
)

# Cleanup SigExt
del sigext_model, sigext_tokenizer
clear_gpu_memory()

print("\n" + "="*60)
print("PHASE 2: GENERATION PHASE")
print("="*60)

clear_gpu_memory()

# Load LLM
llm_model, llm_tokenizer = load_llm(
    GLOBAL_CONFIG["llm_model_id"],
    QUANT_CONFIG
)

llm_terminators = [
    llm_tokenizer.eos_token_id,  # <|end_of_text|>
    llm_tokenizer.convert_tokens_to_ids("<|eot_id|>") # <|eot_id|>
]

# Define generation arguments
gen_kwargs = {
    "max_new_tokens": 512,
    "do_sample": GLOBAL_CONFIG["do_sample"],
    "eos_token_id": llm_terminators,
}

if GLOBAL_CONFIG["do_sample"]:
    gen_kwargs.update({
        "temperature": GLOBAL_CONFIG['temperature'],
        "repetition_penalty": GLOBAL_CONFIG['repetition_penalty'],
        "top_p": GLOBAL_CONFIG['top_p']
    })

# Create text generation pipeline for summary
gen_pipe = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=llm_tokenizer,
    **gen_kwargs
)

llm_summary = HuggingFacePipeline(pipeline=gen_pipe)


# Create chains
summary_prompt = PromptTemplate(
    template=SUMMARY_PROMPT,
    input_variables=["source", "keyphrases"]
)
summary_chain = summary_prompt | llm_summary | StrOutputParser()

data_with_summaries = run_generation_phase(processed_data, summary_chain)

del llm_model, llm_tokenizer, gen_pipe, gen_kwargs, summary_chain
clear_gpu_memory()

# save summaries
output_filename = "generation_output.json"
with open(output_filename, "w") as f:
    json.dump(data_with_summaries, f)
print(f"Data saved as {output_filename}.json ")

## 6. Cleanup

In [ ]:
# Cleanup
del llm_model, llm_tokenizer, gen_pipe, llm_summary
clear_gpu_memory()

print(" Cleanup complete!")